# FIN8311 : le modèle à 3 facteurs de Fama-French est-il stable en crise et en calme ?

**Équipe** : Équipe FIN8311

## Problématique
Le modèle de Fama-French explique le rendement d'un panier d'actions avec 3 facteurs : le marché (`Mkt-RF`), la taille (`SMB`) et la valeur (`HML`).
Avec peu de mois de données, les coefficients β estimés par les moindres carrés (OLS) peuvent être instables d'une période à l'autre.

**Question** : les β et les facteurs retenus changent-ils entre une période de **crise** et une période de **calme** ? Ridge et Lasso, qui pénalisent les β, donnent-ils des résultats plus stables ou plus lisibles que l'OLS ?

## Plan du notebook
1. Outils (imports)
2. Chargement des deux fichiers de données et fusion
3. Construction des rendements excédentaires (les `y`)
4. Les deux périodes : crise (22 mois) et calme (22 mois)
5. Régression OLS : les β et leurs erreurs standards
6. Standardisation, Ridge et Lasso à la main
7. Graphique : les chemins du Lasso
8. La fonction `comparer` (validation croisée et règle 1-SE)
9. Tableau final : 6 paniers x 2 périodes
10. Conclusion et limites
11. Pistes de robustesse (optionnel)

## Données (à télécharger, non incluses dans ce dépôt)
Source : [bibliothèque de données de Kenneth French](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)
- `F-F_Research_Data_Factors.csv` (facteurs `Mkt-RF`, `SMB`, `HML` et taux sans risque `RF`, mensuel, en %)
- `6_Portfolios_2x3.csv` (6 paniers taille x valeur, rendements mensuels pondérés par la valeur, en %)

## Comment obtenir les résultats (copier-coller ou ouvrir dans Colab)
1. Télécharger les deux fichiers CSV ci-dessus (ils sont dans les fichiers ZIP du site, section « Fama/French 3 Factors » et « 6 Portfolios Formed on Size and Book-to-Market (2 x 3) »).
2. Ouvrir ce notebook dans Google Colab (colab.research.google.com, *Fichier → Importer le notebook*), ou copier-coller les cellules dans un nouveau notebook.
3. Cliquer sur *Exécution → Tout exécuter*. À la cellule de la section 2, Colab demande les deux fichiers : les sélectionner.
4. Les tableaux et graphiques s'affichent sous chaque cellule.

## 1. Outils

In [ ]:
import numpy as np                      # calculs sur des listes de nombres
import pandas as pd                     # tableaux (comme Excel)
import matplotlib.pyplot as plt         # graphiques
import statsmodels.api as sm            # régression OLS avec erreurs standards
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, RidgeCV, LassoCV, LinearRegression, lasso_path

## 2. Chargement et fusion
- `skiprows` saute les lignes de texte en haut du fichier.
- `nrows=1203` ne lit que les 1203 mois (juillet 1926 à septembre 2026) et ignore le tableau annuel placé plus bas.
- La colonne des dates n'a pas de titre : pandas l'appelle `Unnamed: 0`, on la renomme `date` (format AAAAMM).

In [ ]:
import os
# Si les fichiers ne sont pas déjà là, Colab ouvre une fenêtre pour les sélectionner
if not (os.path.exists("F-F_Research_Data_Factors.csv") and os.path.exists("6_Portfolios_2x3.csv")):
    try:
        from google.colab import files
        print("Sélectionnez les deux fichiers CSV :")
        files.upload()
    except ImportError:
        print("Placez les deux fichiers CSV dans le même dossier que ce notebook.")

In [ ]:
df = pd.read_csv("F-F_Research_Data_Factors.csv", skiprows=4, nrows=1203)
df = df.rename(columns={"Unnamed: 0": "date"})

pan = pd.read_csv("6_Portfolios_2x3.csv", skiprows=15, nrows=1203)
pan = pan.rename(columns={"Unnamed: 0": "date"})

# On colle les deux tableaux mois par mois (comme RECHERCHEV dans Excel)
data = pd.merge(df, pan, on="date")
print(data.shape)        # attendu : (1203, 11)
data.head()

## 3. Les `y` : rendements excédentaires
`Mkt-RF` est déjà « moins le taux sans risque ». On fait pareil pour chaque panier : **rendement du panier − RF**.

In [ ]:
paniers = ["SMALL LoBM", "ME1 BM2", "SMALL HiBM", "BIG LoBM", "ME2 BM2", "BIG HiBM"]

for p in paniers:
    data[p + "_exc"] = data[p] - data["RF"]

facteurs = ["Mkt-RF", "SMB", "HML"]
data.tail()

## 4. Crise et calme
- **Crise** : récessions américaines officielles (NBER) : 2007-12 à 2009-06 (19 mois) et 2020-02 à 2020-04 (3 mois) = **22 mois**.
- **Calme** : 2013-01 à 2014-10 = **22 mois**, pour comparer avec la même quantité de données.
- `&` veut dire ET, `|` veut dire OU.

In [ ]:
crise = data[((data["date"] >= 200712) & (data["date"] <= 200906)) |
             ((data["date"] >= 202002) & (data["date"] <= 202004))]
calme = data[(data["date"] >= 201301) & (data["date"] <= 201410)]

print(crise.shape[0], calme.shape[0])   # attendu : 22 22

## 5. Régression OLS
`y = α + β₁·Mkt-RF + β₂·SMB + β₃·HML + erreur`.
- `params` donne les coefficients (α, β).
- `bse` donne leurs **erreurs standards** (l'incertitude). Règle de pouce : zone probable du vrai β ≈ β ± 2 × erreur standard.

Exemple détaillé avec le panier `SMALL HiBM`, puis boucle sur les 6 paniers.

In [ ]:
y = data["SMALL HiBM_exc"]
X = sm.add_constant(data[facteurs])        # ajoute la colonne de 1 : c'est elle qui donne alpha
r_tout = sm.OLS(y, X).fit()

yc = crise["SMALL HiBM_exc"]
r_crise = sm.OLS(yc, sm.add_constant(crise[facteurs])).fit()

yk = calme["SMALL HiBM_exc"]
r_calme = sm.OLS(yk, sm.add_constant(calme[facteurs])).fit()

resume = pd.DataFrame({
    "beta_tout": r_tout.params, "se_tout": r_tout.bse,
    "beta_crise": r_crise.params, "se_crise": r_crise.bse,
    "beta_calme": r_calme.params, "se_calme": r_calme.bse,
}).round(3)
resume

In [ ]:
# Même chose pour les 6 paniers : beta en crise et en calme
lignes = []
for p in paniers:
    for nom, d in [("crise", crise), ("calme", calme)]:
        r = sm.OLS(d[p + "_exc"], sm.add_constant(d[facteurs])).fit()
        lignes.append([p, nom] + list(r.params.round(3)) + list(r.bse.round(3)))
colonnes = ["panier", "periode", "alpha", "b_Mkt", "b_SMB", "b_HML", "se_alpha", "se_Mkt", "se_SMB", "se_HML"]
tableau_ols = pd.DataFrame(lignes, columns=colonnes)
tableau_ols

## 6. Standardiser, puis Ridge et Lasso à la main
- **Standardiser** met les facteurs sur la même échelle (moyenne 0, écart-type 1), pour que la pénalité traite tous les facteurs de la même façon.
- `alpha` est la force de la pénalité. Quand `alpha` monte, les β de Ridge rétrécissent ; ceux du Lasso rétrécissent puis deviennent **exactement 0**.

In [ ]:
Xs = StandardScaler().fit_transform(crise[facteurs])
Xk = StandardScaler().fit_transform(calme[facteurs])

print("Lasso, période de CRISE (colonnes : Mkt-RF, SMB, HML)")
for a in [0.5, 1, 2, 3, 4, 6, 8]:
    print(a, Lasso(alpha=a).fit(Xs, yc).coef_.round(2))

print("Lasso, période de CALME")
for a in [0.1, 0.3, 0.5, 0.8, 1, 1.5, 2, 3]:
    print(a, Lasso(alpha=a).fit(Xk, yk).coef_.round(2))

## 7. Graphique : les chemins du Lasso
Un chemin = la valeur de chaque β quand la pénalité augmente. Le facteur qui tombe à zéro en dernier est le plus solide.
Attention : les deux graphiques n'ont pas la même échelle horizontale ; on compare l'**ordre** des chutes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (Xx, yy, titre) in zip(axes, [(Xs, yc, "Crise"), (Xk, yk, "Calme")]):
    al, coefs, _ = lasso_path(Xx, yy - yy.mean())   # y centré : lasso_path n'ajoute pas d'ordonnée à l'origine
    for i, f in enumerate(facteurs):
        ax.plot(al, coefs[i], label=f)
    ax.set_title("Chemins du Lasso : " + titre)
    ax.set_xlabel("alpha (force de la pénalité)")
    ax.set_ylabel("beta (facteurs standardisés)")
    ax.legend()
plt.tight_layout()
plt.savefig("chemins_lasso.png", dpi=150)
plt.show()

## 8. La fonction `comparer`
Une fonction (`def`) est une recette réutilisable. Pour un tableau `d` (crise ou calme) et un panier, elle estime :
- **OLS** (sans pénalité) ;
- **Ridge** (`RidgeCV` choisit alpha par validation croisée efficace) ;
- **Lasso** (`LassoCV` choisit alpha par validation croisée en `cv` morceaux) ;
- **Lasso 1-SE** : le plus grand alpha dont l'erreur reste dans une erreur standard du meilleur. Avec peu de données, le « meilleur » alpha est incertain ; on prend le modèle le plus simple presque aussi bon.

In [ ]:
def comparer(d, panier, cv=5):
    X = StandardScaler().fit_transform(d[facteurs])
    y = d[panier + "_exc"]
    ols   = LinearRegression().fit(X, y)
    ridge = RidgeCV(alphas=np.logspace(-2, 3, 60)).fit(X, y)
    lcv   = LassoCV(cv=cv).fit(X, y)
    moy   = lcv.mse_path_.mean(axis=1)                       # erreur moyenne de test, pour chaque alpha
    marge = lcv.mse_path_.std(axis=1) / np.sqrt(cv)          # incertitude de cette erreur
    i     = moy.argmin()                                     # position du meilleur alpha
    a_1se = lcv.alphas_[moy <= moy[i] + marge[i]].max()      # règle 1-SE
    l1    = Lasso(alpha=a_1se).fit(X, y)
    print("alpha : Ridge =", round(ridge.alpha_, 2), "| Lasso min =", round(lcv.alpha_, 3), "| Lasso 1-SE =", round(a_1se, 3))
    return pd.DataFrame({"OLS": ols.coef_, "Ridge": ridge.coef_,
                         "Lasso_min": lcv.coef_, "Lasso_1SE": l1.coef_}, index=facteurs).round(2)

# Exemple : un panier, deux périodes
print("SMALL HiBM, crise");  display(comparer(crise, "SMALL HiBM"))
print("SMALL HiBM, calme");  display(comparer(calme, "SMALL HiBM"))

## 9. Tableau final : quels facteurs le Lasso (1-SE) élimine-t-il ?
Deux boucles imbriquées : 6 paniers x 2 périodes = 12 combinaisons.
Un facteur dont le β vaut exactement 0 est « éliminé ». On cherche les paniers dont la liste change entre crise et calme.

In [ ]:
lignes = []
for p in paniers:
    for nom, d in [("crise", crise), ("calme", calme)]:
        t = comparer(d, p)
        elimines = ""
        for f in facteurs:
            if t.loc[f, "Lasso_1SE"] == 0:
                elimines = elimines + f + " "
        lignes.append([p, nom, elimines])

final = pd.DataFrame(lignes, columns=["panier", "periode", "elimines_par_Lasso_1SE"])
final

## 10. Conclusion et limites

**Ce que l'on observe (d'après nos exécutions)**
- Le marché (`Mkt-RF`) domine dans les deux périodes pour les 6 paniers.
- Les β de l'OLS bougent peu entre crise et calme pour la plupart des paniers. Pour `SMALL HiBM`, `SMB` et `HML` changent de rang d'une période à l'autre, mais leurs zones β ± 2 erreurs standards se chevauchent.
- Ridge et Lasso donnent des β proches de l'OLS dans la plupart des cas.
- Le Lasso (1-SE) élimine souvent `SMB` pour les paniers de grandes entreprises, jamais pour les petites. `BIG LoBM` est le cas qui change selon la période.

**Limites**
- 22 mois par période : β incertains.
- 3 facteurs seulement.
- Les facteurs SMB et HML sont fabriqués à partir des mêmes paniers (taille x valeur).
- Un seul réglage de validation croisée (`cv=5`) et une seule fenêtre de calme.
- Le choix des dates de crise est un choix humain (dates NBER).

> Les chiffres exacts sont ceux affichés par les cellules ci-dessus. Relancer le notebook avec les vrais fichiers pour les voir.

## 11. Pistes de robustesse (optionnel : pas encore fait)
Refaire le tableau final avec (a) un calme plus long (2013-01 à 2019-12, 84 mois) et (b) `cv=4` et `cv=6`, pour voir si les facteurs éliminés changent.

In [ ]:
calme_long = data[(data["date"] >= 201301) & (data["date"] <= 201912)]
print(calme_long.shape[0])   # attendu : 84

def tableau_final(periodes, cv=5):
    lignes = []
    for p in paniers:
        for nom, d in periodes:
            t = comparer(d, p, cv=cv)
            elimines = ""
            for f in facteurs:
                if t.loc[f, "Lasso_1SE"] == 0:
                    elimines = elimines + f + " "
            lignes.append([p, nom, elimines])
    return pd.DataFrame(lignes, columns=["panier", "periode", "elimines_par_Lasso_1SE"])

# Décommenter pour lancer (les messages 'alpha : ...' s'affichent pour chaque cas)
# tableau_final([("crise", crise), ("calme_long", calme_long)])
# tableau_final([("crise", crise), ("calme", calme)], cv=4)
# tableau_final([("crise", crise), ("calme", calme)], cv=6)